In [220]:
import polars as pl

In [221]:
pl.Config(tbl_rows=200)
pl.Config.set_tbl_width_chars(200)

polars.config.Config

In [222]:
zdroj = pl.read_parquet(
    "data/cro.parquet"
).filter(
    pl.col('since').dt.year().is_in([2022,2023,2024])
).with_columns(
    pl.col("track").str.to_lowercase()
).with_columns(
    pl.col("interpret").str.to_lowercase()
).with_columns(
    (pl.col('interpret') + ": " + pl.col("track")).alias("banger")
).with_columns(
    pl.col('since').dt.week().alias('tyden')
)

In [223]:
zdroj.sort(by='since')

since,id,interpret,track,track_id,stanice,interpret_id,itemcode,banger,tyden
"datetime[μs, UTC]",str,str,str,str,cat,str,str,str,i8
2022-01-01 00:01:22 UTC,"""12426056""","""slade""","""my oh my""","""7760""","""dvojka""","""3173""","""9000537""","""slade: my oh my""",52
2022-01-01 00:02:05 UTC,"""12426058""","""freetown quartet""","""sub nights""","""84604""","""jazz""","""25233""","""GS08929""","""freetown quartet: sub nights""",52
2022-01-01 00:02:58 UTC,"""12426063""","""virginia wing""","""the second shift""","""50138""","""radiowave""","""14874""","""9776451""","""virginia wing: the second shif…",52
2022-01-01 00:06:06 UTC,"""12426069""","""bugmen""","""out on the weekend""","""676""","""radiowave""","""622""","""9772254""","""bugmen: out on the weekend""",52
2022-01-01 00:06:08 UTC,"""12426070""","""james blunt""","""1973""","""7631""","""radiozurnal""","""3118""","""1007031""","""james blunt: 1973""",52
2022-01-01 00:07:39 UTC,"""12426073""","""slowly rolling camera""","""protagonist""","""23862""","""jazz""","""8074""","""GS06101""","""slowly rolling camera: protago…",52
2022-01-01 00:08:37 UTC,"""12426075""","""cloud control""","""scar""","""20476""","""radiowave""","""7114""","""9773977""","""cloud control: scar""",52
2022-01-01 00:12:23 UTC,"""12426080""","""baynk""","""esther ft. tinashe""","""92021""","""radiowave""","""20098""","""9778380""","""baynk: esther ft. tinashe""",52
2022-01-01 00:12:53 UTC,"""12426081""","""louis sclavis""","""archéologie""","""2281""","""jazz""","""944""","""GS02788""","""louis sclavis: archéologie""",52


## Skladby

In [225]:
tri_roky = zdroj.with_columns(
    pl.col('since').dt.year().alias('rok')
).group_by(
    'banger'
).agg(
    pl.col('rok').unique().len()
).filter(
    pl.col('rok') == 3
).select(
    pl.col('banger')
).unique().to_series().to_list()

In [226]:
letni = zdroj.filter(pl.col('banger').is_in(tri_roky)).filter(pl.col("since").dt.month().is_in([7,8])).group_by('banger').len().rename({'len':'v_lete'})
zimni = zdroj.filter(pl.col('banger').is_in(tri_roky)).filter(pl.col("since").dt.month().is_in([1,2,3,4,5,6,9,10,11,12])).group_by('banger').len().rename({'len':'v_zime'})

In [227]:
pomer = letni.join(zimni, how='full', on='banger').with_columns(
    pl.when(
        pl.col('banger').is_null()
    ).then(
        pl.col('banger_right')
    ).otherwise(
        pl.col('banger')
    ).alias(
        'banger'
    )
).with_columns(
    pl.col('v_lete').fill_null(0)
).drop(
    'banger_right'
).with_columns(
    (pl.col('v_lete') / pl.col('v_zime')).alias('pomer')
)

In [262]:
pomer.filter(
    pl.col('v_lete') > 8
).sort(
    by='pomer',descending=True
)

banger,v_lete,v_zime,pomer
str,u32,u32,f64
"""karel gott: léto jak má být""",17,1,17.0
"""hana zagorová: ahoj, léto""",21,3,7.0
"""shawn mendes, camila cabello: …",23,10,2.3
"""claire rosinkranz: 123""",43,21,2.047619
"""adam baldych quintet & agata s…",17,9,1.888889
"""righeira: vamos a la playa""",15,8,1.875
"""julius rodriguez: blues at the…",27,15,1.8
"""adam baldych quintet & agata s…",19,12,1.583333
"""los del rio: macarena""",17,11,1.545455


In [270]:
letni_bengry = pomer.filter(
    pl.col('v_lete') > 8
).sort(
    by='pomer',descending=True
).filter(
    pl.col("pomer") >= 1
).select(pl.col("banger")).to_series().to_list()

In [274]:
alt.Chart(
    zdroj.filter(pl.col('banger').is_in(letni_bengry)).group_by(['banger','tyden']).len()
).mark_circle(
).encode(
    alt.X('tyden:Q'),
    alt.Y('banger:N'),
    alt.Size('len:Q')
)

alt.Chart(...)

In [229]:
vyber_strojovy = pomer.filter(
    pl.col('v_lete') > 20
).sort(
    by='pomer',descending=True
).head(15).select(pl.col('banger')).to_series().to_list()

In [230]:
pomer.filter(
    pl.col('v_lete') > 20
).sort(
    by='pomer',descending=True
).select(pl.col('banger')).to_series().to_list()[1]

'shawn mendes, camila cabello: seňorita'

In [231]:
pomer.filter(pl.col('pomer').is_null()).sort(by='v_lete',descending=True)

banger,v_lete,v_zime,pomer
str,u32,u32,f64
"""dan bárta: superstar""",4,null,null
"""hradišťan: krátký popis léta""",3,null,null
"""marcela laiferová: hrdá láska""",3,null,null
"""alizée: a quoi reve une jeune …",3,null,null
"""quattro orchestra, dir. marek …",3,null,null
"""food: tobiko""",3,null,null
"""gipsy.cz: jižní spojka""",3,null,null


In [232]:
pomer.filter(pl.col('pomer').is_null()).sort(by='v_lete',descending=True).select(pl.col('banger')).to_series().to_list()[2]

'marcela laiferová: hrdá láska'

In [233]:
import altair as alt

In [234]:
rucni_vyber = [
    "EWA FARNA: ZKRÁCENO V PŘEKLADU",
    'CHAI: Hero Journey ft. Superorganism',
    "CLAIRE ROSINKRANZ: 123",
    "CHINASKI: DVĚ SRDCE VEJPŮL",
    "TELENOVA: Haunted",
    "KAREL ZICH: LÉTO JAK MÁ BÝT",
    "HANA ZAGOROVÁ: AHOJ, LÉTO",
    'HELENA VONDRÁČKOVÁ: LÉTO JE LÉTO',
]

In [235]:
graf = zdroj.filter(pl.col('banger').is_in(vyber_strojovy)).group_by(['banger','tyden']).len()
graf

banger,tyden,len
str,i8,u32
"""brandon allen: and satisfy""",45,1
"""telenova: haunted""",50,1
"""flohio: spf""",42,3
"""flohio: spf""",47,1
"""flohio: spf""",34,16
"""latto: pussy""",38,2
"""latto: pussy""",37,2
"""telenova: haunted""",29,5
"""flohio: spf""",19,1


In [236]:
alt.Chart(
    graf
).mark_circle(
).encode(
    alt.X('tyden:Q'),
    alt.Y('banger:N'),
    alt.Size('len:Q')
)

alt.Chart(...)

## Interpreti

In [238]:
tri_roky_i = zdroj.with_columns(
    pl.col('since').dt.year().alias('rok')
).group_by(
    'interpret'
).agg(
    pl.col('rok').unique().len()
).filter(
    pl.col('rok') == 3
).select(
    pl.col('interpret')
).unique().to_series().to_list()

In [239]:
letni_i = zdroj.filter(pl.col('interpret').is_in(tri_roky_i)).filter(pl.col("since").dt.month().is_in([7,8])).group_by('interpret').len().rename({'len':'v_lete'})
zimni_i = zdroj.filter(pl.col('interpret').is_in(tri_roky_i)).filter(pl.col("since").dt.month().is_in([1,2,3,4,5,6,9,10,11,12])).group_by('interpret').len().rename({'len':'v_zime'})

In [240]:
pomer_i = letni_i.join(zimni_i, how='full', on='interpret').with_columns(
    pl.when(
        pl.col('interpret').is_null()
    ).then(
        pl.col('interpret_right')
    ).otherwise(
        pl.col('interpret')
    ).alias(
        'interpret'
    )
).with_columns(
    pl.col('v_lete').fill_null(0)
).drop(
    'interpret_right'
).with_columns(
    (pl.col('v_lete') / pl.col('v_zime')).alias('pomer')
)

In [241]:
pomer_i.filter(pl.col('v_lete') >= 10).sort(by='pomer',descending=True)

interpret,v_lete,v_zime,pomer
str,u32,u32,f64
"""shawn mendes, camila cabello""",23,10,2.3
"""righeira""",15,8,1.875
"""julius rodriguez""",71,50,1.42
"""yung hurn""",35,32,1.09375
"""adam baldych quintet & agata s…",55,56,0.982143
"""telenova""",45,47,0.957447
"""lucie bílá & jan bendig""",16,18,0.888889
"""david dorůžka & piotr wylezol …",66,81,0.814815
"""los del rio""",17,21,0.809524
